# Deliverable 2 -- Sistema completo en Google Colab (T4)

Este notebook es la versión **final y ejecutable** del flujo de `prueba_d2.ipynb` (D1).
Corre el sistema de extremo a extremo sobre **Benchmark2 (45 frases, held-out)**:

```
mensaje ──> [1] Qwen3-4B + LoRA (fine-tune)   -> campos semanticos (SIN fecha)
        ──> [2] resolvedor_temporal.py        -> fecha (determinista)
        ──> [3] normalizacion.py              -> contrato de 9 campos
        ──> [4] calendario/calendar_api.py    -> evento + .ics
```

**Brazos que se corren** (mismo criterio de corrección: `evaluar.py` de D1):

| Brazo | Extracción | Fecha | Qué mide |
|---|---|---|---|
| A | prompt directo | modelo | baseline |
| B | prompt estructurado + few-shot | modelo | la mejora de D1 |
| C | prompt estructurado + tool | tool | aporte de la herramienta |
| D | **fine-tune + tool** | tool | **solución final** |

---

## Cómo llegar a Colab con todo

1. **Comprime la carpeta del proyecto** (la que contiene `pipeline/`, `benchmark/`,
   `experimentos/`, `calendario/`, `evaluar.py`) en `proyecto.zip`.
2. En Colab: `Archivo → Subir notebook` y elige este `.ipynb`.
3. Activa **GPU T4**: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4`.
4. Ejecuta las celdas en orden. La celda 2 pide `proyecto.zip` (o clona el repo si está en GitHub).
5. Para el **video**, ejecuta la celda `DEMO` con la cámara de pantalla activa.

> **Regla de no fuga**: el fine-tuning se entrenó con `datos/train.jsonl` (sintético).
> Benchmark2 nunca se usó para entrenar; aquí es solo evaluación.

In [ ]:
# 1) Dependencias (T4)
!nvidia-smi
%pip install -q -U "transformers>=4.51.0" datasets accelerate peft "trl==0.21.0" bitsandbytes

In [ ]:
# 2) Traer el proyecto a Colab
#    Opcion A (si el repo esta en GitHub): descomenta
# !git clone https://github.com/<usuario>/SML-GEN-AI-for-Calendar-services.git proyecto
#    Opcion B: sube el zip de la carpeta del proyecto (se genera con datos/preparar_zip.py)
import os, sys, shutil, zipfile

def extraer_zip(zip_path, destino="proyecto"):
    """Extraccion robusta: normaliza separadores Windows (\\) a POSIX (/)."""
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            nombre = info.filename.replace("\\", "/")
            if not nombre or nombre.endswith("/") or "__pycache__" in nombre:
                continue
            ruta = os.path.join(destino, *nombre.split("/"))
            os.makedirs(os.path.dirname(ruta), exist_ok=True)
            with z.open(info) as src, open(ruta, "wb") as dst:
                shutil.copyfileobj(src, dst)

# si falta el paquete pipeline, es que el zip no esta (o se extrajo mal)
if not os.path.isfile(os.path.join("proyecto", "pipeline", "__init__.py")):
    if os.path.isdir("proyecto"):
        shutil.rmtree("proyecto")          # limpieza de una extraccion anterior rota
    from google.colab import files
    print("Sube proyecto.zip (raiz del proyecto comprimida):")
    subido = files.upload()
    zip_path = next(iter(subido))
    extraer_zip(zip_path)
    # si el zip tenia una carpeta raiz unica, la bajamos un nivel
    hijos = [h for h in os.listdir("proyecto") if h != "__MACOSX"]
    if len(hijos) == 1 and os.path.isdir(os.path.join("proyecto", hijos[0])):
        raiz_zip = os.path.join("proyecto", hijos[0])
        for h in os.listdir(raiz_zip):
            shutil.move(os.path.join(raiz_zip, h), os.path.join("proyecto", h))
        os.rmdir(raiz_zip)

RAIZ = os.path.abspath("proyecto")
sys.path.insert(0, RAIZ)
assert os.path.isfile(os.path.join(RAIZ, "pipeline", "__init__.py")), "el zip se extrajo mal"
print("Proyecto en:", RAIZ)
print(sorted(os.listdir(RAIZ)))

In [ ]:
# 3) Benchmark (SOLO evaluacion) y configuracion
import json

from pipeline.resolvedor_temporal import FECHA_REF, analizar_temporal
from pipeline import prompts as P

RUTAS_BENCH = [os.path.join(RAIZ, "benchmark", "benchmark2.json"),
               "/content/frases (1).json"]
ruta_bench = next(r for r in RUTAS_BENCH if os.path.exists(r))
with open(ruta_bench, encoding="utf-8") as f:
    BENCH = json.load(f)["frases"]

MODELO_ID = "Qwen/Qwen3-4B-Instruct-2507"
FECHA_REFERENCIA = P.FECHA_REFERENCIA
SEMILLA = 42

print(f"Modelo: {MODELO_ID}")
print(f"Fecha de referencia: {P.DIA_REFERENCIA} {FECHA_REFERENCIA}")
print(f"Benchmark2: {len(BENCH)} frases (held-out, nunca usadas en el fine-tuning)")

In [ ]:
# 4) Modelo base + ADAPTER LoRA fine-tuneado
#    Pon en ADAPTER la ruta del adapter que salio de entrenamiento/qlora_colab.ipynb:
#      "qwen3-4b-semantic-lora"                     (si esta en la sesion)
#      "/content/drive/MyDrive/qwen3-4b-semantic-lora"
#      "usuario/qwen3-4b-semantic-lora"             (HF Hub)
#    Si lo dejas vacio se usa el modelo base (sin fine-tune) y se avisa.

ADAPTER = ""   # <-- EDITAR

from pipeline.modelo import GeneradorHF

if ADAPTER:
    print("Cargando base + adapter:", ADAPTER)
else:
    print("AVISO: sin adapter. El brazo D usara el modelo BASE (no es la solucion final).")
generador = GeneradorHF(MODELO_ID, adapter=ADAPTER or None, semilla=SEMILLA)
print("Listo:", generador.modelo_id, "| adapter:", generador.adapter)

In [ ]:
# 5) Pipeline con los 4 brazos
from pipeline.pipeline import BRAZOS, CalendarPipeline

tuberia = CalendarPipeline(generador, fecha_ref=FECHA_REFERENCIA)
print("Brazos disponibles:", BRAZOS)

## 6) Ejecución de los 4 brazos sobre las 45 frases

Cada brazo se corre con **el mismo modelo, el mismo decoding** (temp 0.01, semilla 42)
y el **mismo criterio de corrección** (`evaluar.py` de D1). El resultado se guarda en el
formato que consume `evaluar.py`.

In [ ]:
# 6) Correr brazos A-D  (en Colab T4: ~10-20 min por brazo)
import os
from pipeline.pipeline import CalendarPipeline

RUTA_RESULTADOS = os.path.join(RAIZ, "resultados")
os.makedirs(RUTA_RESULTADOS, exist_ok=True)

BRAZOS_A_CORRER = ["A", "B", "C", "D"]   # puedes acotar para probar

for brazo in BRAZOS_A_CORRER:
    filas = []
    for fr in BENCH:
        evento = tuberia.correr_brazo(brazo, fr["texto"])
        filas.append({
            "id": fr["id"],
            "texto": fr["texto"],
            "resultado_modelo": CalendarPipeline.solo_contrato(evento),
            "entorno": "hf:%s%s" % (MODELO_ID, " +adapter" if ADAPTER else ""),
        })
        print("  [%s] id=%2d  fecha=%s" % (brazo, fr["id"], filas[-1]["resultado_modelo"].get("fecha")))
    ruta = os.path.join(RUTA_RESULTADOS, "resultados_%s.json" % brazo)
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(filas, f, ensure_ascii=False, indent=1)
    print("Guardado:", ruta)

## 7) Evaluación con el criterio idéntico al de D1

`experimentos/evaluar_todo.py` importa `evaluar.py` tal cual (sin reimplementar nada),
calcula exactitud global/por campo/por categoría, y escribe:

- `resultados/resumen.json`
- `resultados/reporte_{brazo}.csv`
- `docs/tabla_resultados.tex` (lista para el documento de una página)

In [ ]:
# 7) Evaluar (criterio de D1) y generar reportes
import subprocess, sys
subprocess.run([sys.executable, os.path.join(RAIZ, "experimentos", "evaluar_todo.py")], check=False)

## DEMO (esto es lo que se graba para el video)

Muestra, **sobre la misma frase**: el baseline (brazo A) y la solución (brazo D),
la corrección que hace la herramienta de fechas, y el evento creado en la API de
calendario. 3 minutos de video salen de aquí.

In [ ]:
# DEMO -- baseline vs solucion en la MISMA frase + evento creado
import json
from calendario.calendar_api import LocalCalendarAPI
from pipeline.pipeline import CalendarPipeline

FRASE = "agenda reunion manana 10am porfa"        # cambiala por una frase dificil si quieres
# FRASE = "q onda, quedamos el viernes a las 7 en lo de siempre?"

api = LocalCalendarAPI(os.path.join(RAIZ, "resultados", "calendario.json"))

print("=" * 70)
print("FRASE:", FRASE)
print("=" * 70)

print("\n[A] BASELINE -- prompt directo, fecha calculada por el modelo:")
a = tuberia.baseline(FRASE)
print(json.dumps(CalendarPipeline.solo_contrato(a), ensure_ascii=False, indent=1))

print("\n[D] SOLUCION -- extractor fine-tuneado + herramienta de fechas:")
d = tuberia.solucion(FRASE)
print(json.dumps(CalendarPipeline.solo_contrato(d), ensure_ascii=False, indent=1))

if a.get("fecha") != d.get("fecha"):
    print("\n  >> la herramienta corrige la fecha: %r -> %r" % (a.get("fecha"), d.get("fecha")))
print("  >> expresion temporal detectada:", analizar_temporal(FRASE, FECHA_REFERENCIA)["fecha_expr"])

registro = api.create_event(CalendarPipeline.solo_contrato(d))
print("\nEvento creado en la API de calendario:", registro["id"])
print(json.dumps(registro, ensure_ascii=False, indent=1))

ics = api.exportar_ics(os.path.join(RAIZ, "resultados", "calendario.ics"))
print("\nICS exportado:", ics)

In [ ]:
# 8) Bajar resultados a tu maquina
from google.colab import files
import glob

for ruta in sorted(glob.glob(os.path.join(RAIZ, "resultados", "*"))):
    print("subiendo", ruta)
    files.download(ruta)
for ruta in sorted(glob.glob(os.path.join(RAIZ, "docs", "tabla_resultados.tex"))):
    files.download(ruta)

## Checklist para el entregable

- [ ] Los 4 brazos corridos con el modelo real (n=45) → `resultados/resultados_*.json`
- [ ] `evaluar_todo.py` muestra exactitud de A vs D (evidencia de mejora)
- [ ] DEMO corrida (video de ≤3 min, link abierto)
- [ ] `docs/tabla_resultados.tex` pegado en `docs/deliverable2.tex`
- [ ] Repo público con README que explique cómo reproducir esto

**Modelo comprometido**: `Qwen/Qwen3-4B-Instruct-2507` (4.0B, Apache 2.0) + adapter
LoRA propio (artefacto derivado declarado). **Intervención**: fine-tuning del extractor
semántico (ataca el fallo de formato/inversión) + herramienta determinista de fechas
(ataca el piso de error en fechas relativas). Vínculo explícito con el diagnóstico de D1.